# 1. Importación de Librerías y Funciones
#
Importamos las librerías necesarias para el modelado y la evaluación, incluyendo:
- `scanpy` y `pandas` para la manipulación de datos.
 - `sklearn` para el modelado (RandomForest, train_test_split, métricas).
- `joblib` para guardar nuestros modelos entrenados.
 - Nuestra función de ploteo personalizada desde la carpeta `src`.

In [ ]:
import scanpy as sc
import pandas as pd
import os
import sys

# Librerías de Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sys.path.append('../src')
from models import train_and_evaluate_model
from plotting import plot_confusion_matrix
from counts_to_tpm import counts_to_tpm

# 2. Carga del Dataset Procesado

Cargamos el dataset limpio y anotado que generamos en el notebook anterior. También definimos las rutas de salida para guardar las figuras y los modelos que generemos.


Definimos las rutas relativas

In [ ]:
DATA_PROCESSED_PATH = '../data/processed/'
PROCESSED_FILENAME = 'lung_cancer_processed_for_modeling.h5ad'
OUTPUTS_PATH = '../outputs/'
FIGURES_PATH = os.path.join(OUTPUTS_PATH, 'figures')
MODELS_PATH = os.path.join(OUTPUTS_PATH, 'models')

Creamos las carpetas de salida si no existen

In [ ]:
os.makedirs(FIGURES_PATH, exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)

Cargamos los datos

In [ ]:
adata_final = sc.read_h5ad(os.path.join(DATA_PROCESSED_PATH, PROCESSED_FILENAME))

print("Dataset procesado cargado exitosamente:")
print(adata_final)

# Experimento 1: Modelo Base con Todos los Genes

 Nuestro primer objetivo es establecer un rendimiento de referencia (baseline). Entrenaremos un clasificador Random Forest utilizando todos los genes disponibles como características para ver cómo de bien puede distinguir los tipos celulares sin ninguna optimización.

In [ ]:
print("\n--- INICIANDO EXPERIMENTO 1: Modelo Base con Todos los Genes ---")

1. Preparar los datos X (features) e y (target)

In [ ]:
X_all_genes = adata_final.X
y = adata_final.obs['cell_type']

2. Dividir en set de entrenamiento y test

In [ ]:
X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all_genes,
    y,
    test_size=0.2,
    random_state=42,  # Usar un estado fijo para reproducibilidad
    stratify=y        # Esencial para mantener la proporción de clases
)

3. Entrenar el modelo

In [ ]:
model_rf_all, report_rf_all, cm_rf_all, classes_rf_all = train_and_evaluate_model(
    'rf',
    X_train_all, y_train,
    X_test_all, y_test,
    output_dir='../outputs',
    model_name='rf'
)

4. Evaluar el modelo

In [ ]:
print("\nReporte de Clasificación (Random Forest):")
print(report_rf_all)

5. Visualizar y guardar resultados

In [ ]:
plot_confusion_matrix(cm_rf_all, classes_rf_all, title='Matriz de Confusión - Random Forest', 
                    save_path='../outputs/figures/cm_rf_all.png')

# Experimento 2: Selección de Características Biológicamente Informada
El primer modelo mostró debilidades, especialmente en la distinción de clases biológicamente similares. Nuestra hipótesis es que el rendimiento puede mejorar si forzamos al modelo a centrarse únicamente en los genes más distintivos de cada tipo celular (genes marcadores).

Usaremos `scanpy.tl.rank_genes_groups` para identificar estos genes.

In [ ]:
print("\n--- INICIANDO EXPERIMENTO 2: Selección de Genes Marcadores ---")

1. Encontrar genes marcadores

In [ ]:
print("Calculando genes marcadores...")
sc.tl.rank_genes_groups(adata_final, groupby='cell_type', method='t-test')

2. Visualizar los marcadores para confirmar

In [ ]:
print("Visualizando los mejores marcadores...")
sc.pl.rank_genes_groups_dotplot(adata_final, n_genes=4, show=True, save="_marker_genes.png")

3. Extraer la lista de genes marcadores para el modelo

In [ ]:
marker_genes_df = pd.DataFrame(adata_final.uns['rank_genes_groups']['names'])
top_n_genes = 25  # Podemos ajustar este número

marker_genes_list = []
for col in marker_genes_df.columns:
    marker_genes_list.extend(marker_genes_df[col].head(top_n_genes))

marker_genes_list = list(set(marker_genes_list))
print(f"\nSe han seleccionado {len(marker_genes_list)} genes marcadores únicos para el nuevo modelo.")

# Experimento 3: Modelo Optimizado con Genes Marcadores

Ahora, re-entrenaremos el clasificador Random Forest, pero esta vez utilizando únicamente el subconjunto de genes marcadores que acabamos de identificar. Compararemos su rendimiento directamente con el del modelo base.


In [ ]:
print("\n--- INICIANDO EXPERIMENTO 3: Modelo Optimizado con Genes Marcadores ---")

1. Preparar los nuevos datos X (features)

In [ ]:
X_markers = adata_final[:, marker_genes_list].X

2. Dividir los datos (usando la misma `y`, y mismos parámetros de split)

In [ ]:
X_train_markers, X_test_markers, y_train_markers, y_test_markers = train_test_split(
    X_markers,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

3. Entrenar el nuevo modelo

In [ ]:
model_rf_markers, report_rf_markers, cm_rf_markers, classes_rf_markers = train_and_evaluate_model(
    'rf',
    X_train_markers, y_train,
    X_test_markers, y_test,
    output_dir='../outputs',
    model_name='rf_marker_genes'
)

4. Evaluar el modelo optimizado

In [ ]:
print("\nReporte de Clasificación (Random Forest (Marker genes)):")
print(report_rf_markers)

5. Visualizar y guardar resultados

In [ ]:
plot_confusion_matrix(cm_rf_markers, classes_rf_markers, title='Matriz de Confusión - RF (Genes Marcadores)', 
                    save_path='../outputs/figures/cm_rf_marker_genes.png')

# Experimento 4: Benchmark con XGBoost
Para asegurar que hemos elegido el mejor enfoque, comparamos nuestro Random Forest con XGBoost, un algoritmo de gradient boosting a menudo más potente.

In [ ]:
print("\n--- INICIANDO EXPERIMENTO 4: Modelo XGBoost con Genes Marcadores ---")

1. Entrenar el modelo XGBoost

In [ ]:
model_xgb_obj, report_xgb, cm_xgb, classes_xgb = train_and_evaluate_model(
    'xgb',
    X_train_markers, y_train,
    X_test_markers, y_test,
    output_dir='../outputs',
    model_name='xgb_marker_genes'
)

2. Evaluar el modelo XGBoost

In [ ]:
print("\nReporte de Clasificación (XGBoost):")
print(report_xgb)

3. Visualizar y guardar resultados

In [ ]:
plot_confusion_matrix(cm_xgb, classes_xgb, title='Matriz de Confusión - XGBoost (Genes Marcadores)',
                    save_path='../outputs/figures/cm_xgb_marker_genes.png')

# Experimento 5: Benchmark con MLP

print("\n--- INICIANDO EXPERIMENTO 4: Modelo MLP con Genes Marcadores ---")

1. Entrenar el modelo MLP

In [ ]:
model_mlp, report_mlp, cm_mlp,classes_mlp = train_and_evaluate_model(
    'mlp',
    X_train_markers, y_train,
    X_test_markers, y_test,
    output_dir='../outputs',
    model_name='mlp_marker_genes',
    hidden_layer_sizes=(128, 64, 32), # Determinar hiperparámetros que pasar
    max_iter=500 
)

2. Evaluar el modelo MLP

In [ ]:
print("\nReporte de Clasificación (MLP):")
print(report_mlp)

3. Visualizar y guardar resultados

In [ ]:
plot_confusion_matrix(cm_mlp, classes_mlp, title='Matriz de Confusión - MLP (Genes Marcadores)',
                    save_path='../outputs/figures/cm_mlp_marker_genes.png')

# 4. Comparación Final y Selección de Modelos

En esta sección final, realizamos una comparación directa de los tres modelos entrenados sobre el subconjunto de **genes marcadores** para seleccionar el clasificador más robusto y equilibrado. La elección no se basará únicamente en la precisión general, sino en una evaluación de su rendimiento en las clases más desafiantes y en su capacidad general para capturar la diversidad celular.

### Tabla Comparativa de Métricas Clave

| Métrica / Modelo                | Random Forest (RF) | XGBoost (XGB)     | MLP (Red Neuronal) |
| :------------------------------ | :----------------: | :---------------: | :----------------: |
| **Precisión (Accuracy)**        | 95.94%             | 96.12%            | **96.24%**         |
| **Macro Avg F1-Score**          | 0.92               | **0.93**          | **0.93**           |
| **F1-Score `epithelial cell`**  | 0.53               | **0.59**          | **0.59**           |
| **F1-Score `pDC`**              | 0.90               | **0.95**          | **0.95**           |

### Análisis de los Resultados

1.  **Justificación de la Selección de Genes Marcadores:**
    El entrenamiento de un modelo inicial con todos los genes (más de 30,000) reveló un rendimiento aparentemente alto, pero con debilidades críticas en la identificación de poblaciones raras y biológicamente similares. La hipótesis fue que la gran cantidad de genes no informativos ("ruido") estaba dificultando que el modelo se enfocara en las señales transcripcionales distintivas. Por ello, se implementó una estrategia de selección de características, utilizando `scanpy.tl.rank_genes_groups` para identificar un subconjunto de genes marcadores altamente específicos para cada tipo celular. El objetivo era forzar a los modelos a aprender de las características más relevantes, mejorando su rendimiento en los casos más difíciles.

2.  **Rendimiento General y Clases Bien Definidas:**
    Todos los modelos exhiben un rendimiento excepcional en la clasificación de las poblaciones celulares mayoritarias y con perfiles transcripcionales muy distintivos. Clases como `T cell`, `B cell`, `mononuclear phagocyte`, `fibroblast`, `mast cell` y `neutrophil` alcanzan F1-Scores consistentemente altos (≥0.97) en los tres clasificadores. Esto confirma que los genes marcadores seleccionados son altamente efectivos para distinguir los principales linajes celulares inmunitarios y del estroma. Sin embargo, el benchmark revela diferencias clave en las clases más complejas.

3.  **Superioridad de XGBoost y MLP en los Casos Difíciles:**
    Aunque el Random Forest (RF) sirvió como un excelente modelo base, tanto XGBoost como el Perceptrón Multicapa (MLP) ofrecen un rendimiento superior. Ambos logran un **Macro Average F1-Score** más alto (0.93 vs 0.92), reflejando una mejor capacidad para manejar el desequilibrio de clases. El verdadero éxito se observa en las dos poblaciones que identificamos como problemáticas:
    *   **Células Epiteliales (`epithelial cell`):** Tanto XGBoost como MLP aumentaron el F1-Score de 0.53 (RF) a **0.59**. Este salto es significativo y se debe a una mejor distinción con respecto a las células malignas.
    *   **Células Dendríticas Plasmocitoides (`pDC`):** El rendimiento en esta población rara mejoró drásticamente, pasando de un F1-Score de 0.90 (RF) a **0.95** para XGBoost y MLP.

4.  **La Persistencia del "Techo Biológico":**
    Es importante destacar que, a pesar de la mejora, el F1-Score para las células epiteliales sigue siendo moderado (~0.59). Esto no debe interpretarse como un fallo de los modelos, sino como un **hallazgo biológico relevante**. La alta similitud transcripcional entre las células epiteliales del pulmón y las células de adenocarcinoma (que se originan de ellas) impone un límite fundamental a la capacidad de cualquier clasificador basado únicamente en la expresión génica. Este resultado subraya la complejidad del problema y sugiere que para una separación perfecta se requerirían datos de otras modalidades ómicas.

### Decisión Final: Selección del Modelo MLP

Tras analizar los resultados, **seleccionamos el modelo MLP (Perceptrón Multicapa) como el clasificador final** para la siguiente fase del proyecto (deconvolución de datos de bulk RNA-seq).

**Justificación:**

*   **Mejor Rendimiento General:** El MLP obtiene la precisión general más alta y comparte el mejor Macro Average F1-Score con XGBoost, posicionándolo como el modelo con el rendimiento más sólido y equilibrado.
*   **Excelencia en Clases Clave:** Muestra un rendimiento excepcional en las clases difíciles, igualando a XGBoost.
*   **Diversidad Metodológica:** La elección del MLP nos permite incorporar un enfoque de Deep Learning en el TFM, demostrando una exploración más amplia de las técnicas de Machine Learning y añadiendo una capa de sofisticación metodológica al trabajo.

El modelo MLP entrenado con genes marcadores (`mlp_marker_genes.joblib`) será, por tanto, la herramienta que utilizaremos para generar la matriz de firmas genéticas y proceder con el análisis de deconvolución.

# 5. Creación de una Matriz de firmas optimizada

El análisis de deconvolución inicial reveló que la matriz de firmas basada en los "Top N" genes por clase era subóptima debido a la alta colinealidad.

Para solucionar esto, construiremos una nueva matriz de firmas basada en los genes que son **globalmente más importantes** para distinguir entre todas las clases. Utilizaremos un modelo Random Forest entrenado sobre todos los genes disponibles en la referencia para extraer estas puntuaciones de importancia (`feature_importances_`).

## 5.1. Preparación de los Datos de Referencia

Cargamos nuestro objeto `adata_final` (o `adata_ref` del Notebook 4), que contiene los datos log-normalizados y todos los genes.

In [ ]:
final_adata_path = '../data/processed/lung_cancer_processed_for_modeling.h5ad'
adata_final = sc.read_h5ad(final_adata_path)

Usamos los datos log-normalizados de la capa que guardamos antes del escalado

In [ ]:
if 'log_normalized' in adata_final.layers:
    X_full = adata_final.layers['log_normalized']
else:
    X_full = adata_final.X

In [ ]:
y_full = adata_final.obs['cell_type']

print(f"Preparando datos con {X_full.shape[0]} células y {X_full.shape[1]} genes.")

## 5.2. Entrenamiento de Random Forest para obtener la importancia de cada gen

Entrenamos un modelo Random Forest con el único propósito de calcular la importancia de cada gen en la tarea de clasificación.

In [ ]:
print("--- Entrenando Random Forest para la selección de características ---")

In [ ]:
feature_selection_rf = RandomForestClassifier(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

feature_selection_rf.fit(X_full, y_full)
print("Entrenamiento completado.")

## 5.3. Selección de los genes más importantes

Extraemos las puntuaciones de importancia y seleccionamos un número predefinido de los mejores genes.

In [ ]:
# Creamos un DataFrame para manejar fácilmente los resultados
feature_importance_df = pd.DataFrame({
    'gene': adata_final.var.index,
    'importance': feature_selection_rf.feature_importances_
}).sort_values('importance', ascending=False)

In [ ]:
# Seleccionamos el número de genes para nuestra nueva firma
n_top_genes_for_signature = 500
top_signature_genes = feature_importance_df.head(n_top_genes_for_signature)['gene'].tolist()

print(f"\nSe han seleccionado los {n_top_genes_for_signature} genes globalmente más importantes.")


In [ ]:
print("Ejemplo de los 10 genes más importantes:")
# Mapeamos a símbolos de gen para interpretar
top_10_symbols = feature_importance_df.head(10).merge(
    adata_final.var[['gene_name']], left_on='gene', right_index=True
)
display(top_10_symbols)

## 5.4. Creación y Guardado de la Matriz de Firmas Optimizada

Finalmente, construimos la nueva matriz de firmas usando solo estos genes seleccionados y la guardamos para usarla en el Notebook 4. La calcularemos en escala TPM para que esté lista para la deconvolución.

In [ ]:
print("\n--- Creando y guardando la nueva matriz de firmas en escala TPM ---")

In [ ]:
gene_lengths = pd.to_numeric(adata_final.raw.var['feature_length'])
ref_counts_matrix = adata_final.raw.X
# Alinear los genes (longitudes con la matriz de conteos)
common_genes = adata_final.raw.var_names.intersection(gene_lengths.index)
ref_counts_matrix_aligned = ref_counts_matrix[:, adata_final.raw.var_names.isin(common_genes)].copy()
gene_lengths_aligned = gene_lengths[adata_final.raw.var_names[adata_final.raw.var_names.isin(common_genes)]]

In [ ]:
# Normalizar a TPM usando la función robusta
ref_tpm_matrix = counts_to_tpm(ref_counts_matrix_aligned, gene_lengths_aligned)

In [ ]:
#Convertir a DataFrame para el groupby
ref_tpm_df = pd.DataFrame.sparse.from_spmatrix(
    ref_tpm_matrix,
    index=adata_final.obs.index,
    columns=gene_lengths_aligned.index
)
ref_tpm_df['cell_type'] = adata_final.obs['cell_type'].values

In [ ]:
# Calculamos la expresión promedio
signature_matrix_full_tpm = ref_tpm_df.groupby('cell_type').mean().T

In [ ]:
# Filtramos para quedarnos solo con los genes seleccionados por importancia
signature_matrix_optimized = signature_matrix_full_tpm.loc[
    signature_matrix_full_tpm.index.isin(top_signature_genes)
]

In [ ]:
# Guardamos la nueva matriz de firmas
OPTIMIZED_SIGNATURE_FILENAME = 'optimized_signature_matrix.parquet'
optimized_signature_path = os.path.join('../data/processed/', OPTIMIZED_SIGNATURE_FILENAME)

print("\nConvirtiendo la matriz de firmas a formato denso para el guardado...")
signature_matrix_optimized_dense = signature_matrix_optimized.sparse.to_dense()

signature_matrix_optimized_dense.columns = signature_matrix_optimized_dense.columns.astype(str)

signature_matrix_optimized_dense.to_parquet(optimized_signature_path)

print(f"\nMatriz de firmas optimizada con {signature_matrix_optimized.shape[0]} genes guardada en: {optimized_signature_path}")
display(signature_matrix_optimized.head())

# 6. Exportación de la Matriz de Firmas para R

Guardamos la matriz de firmas optimizada en formato TSV para su uso en el pipeline de deconvolución en R.

In [ ]:
print("--- Exportando matriz de firmas a formato TSV para R ---")

# Ruta de salida
SIGNATURE_TSV_FILENAME = 'optimized_signature_for_R.tsv'
signature_tsv_path = os.path.join('../data/processed/', SIGNATURE_TSV_FILENAME)

# Preparar y guardar la matriz de firmas
signature_to_save_r = signature_matrix_optimized.copy()
signature_to_save_r.index.name = 'gene'
signature_to_save_r.to_csv(signature_tsv_path, sep='\t')

print(f"Matriz de firmas guardada en: {signature_tsv_path}")